# OmniXtreme Humanoid Control with OpenVINO

**OmniXtreme** is a unified policy framework for high-dynamic humanoid motion tracking. A flow-matching base policy produces whole-body actions for a Unitree G1 humanoid, a residual policy refines them, and a forward-kinematics (FK) network provides the anchor orientation used in the observation.

This notebook is a **step-by-step manual guide** for running the pretrained OmniXtreme policy in the MuJoCo simulator with OpenVINO: prepare the environment, clone the OmniXtreme repository, apply the OpenVINO patch, and run the sim-to-sim deployment. Follow the markdown sections in order and run each listed command in a **terminal** from the folder indicated in that section. The only Jupyter requirement is reading the notebook itself.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Setup](#Setup)
- [Motion data](#Motion-data)
- [OpenVINO inference](#OpenVINO-inference)
- [ONNX Runtime baseline](#ONNX-Runtime-baseline)
- [Device and precision defaults](#Device-and-precision-defaults)



<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/omnixtreme/omnixtreme.ipynb" />

### Installation Instructions

This directory contains `setup.sh`, `requirements.txt` and `omnix_changes.patch` for preparing the OmniXtreme environment. The setup script checks out the pinned upstream repository, applies the OpenVINO patch to `deploy_mujoco.py`, and installs the required dependencies.

Run the following setup steps in a terminal:

1. Install the system packages and Intel GPU runtime listed in [Prerequisites](#Prerequisites).
2. Create and activate the `omnix314` python virtual environment as shown in [Prerequisites](#Prerequisites).
3. From this (`omnixtreme`) directory, run `./setup.sh` to prepare the OmniXtreme checkout and install its dependencies; see [Setup](#Setup).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

```bash
# Install the required packages specific to this notebook.
sudo apt update && sudo apt install -y \
  git

# Install Intel GPU runtime for OpenCL for using Intel GPU device with OpenVINO
sudo add-apt-repository -y ppa:kobuk-team/intel-graphics
sudo apt update
sudo apt install -y --no-install-recommends \
    libze-intel-gpu1 \
    intel-opencl-icd

# Create a Python 3.14 environment. Conda can be used to manage environments:
conda create -n omnix314 python=3.14
conda activate omnix314
pip install ipykernel nbconvert
# conda deactivate
# conda env remove -n omnix314
```

**Python 3.14** is recommended to run this notebook. The MuJoCo viewer needs a desktop session (or X forwarding); use `--headless` otherwise.

## Setup
[back to top ⬆️](#Table-of-contents:)

Run `setup.sh` in the terminal from this (`omnixtreme`) folder. It will:

- Clone the upstream OmniXtreme repo into `omnix_repo/` (pinned to commit `6f2716d`). It will be skipped if already present.
- Apply `omnix_changes.patch` to `omnix_repo/deploy_mujoco.py`. It will be skipped if already applied.
- Install CPU `torch` from the PyTorch CPU wheel index, plus the remaining Python requirements (`openvino`, `onnxruntime`, `mujoco`, ...).

The upstream repository already contains the exported ONNX graphs in `omnix_repo/policy/` (`fk_trt.onnx`, `base_policy_trt.onnx`, `residual_policy.onnx`) and the G1 robot model in `omnix_repo/robots/g1/`.

## Motion data
[back to top ⬆️](#Table-of-contents:)

Download the curated reference motion `motion.npz` from the [link in the OmniXtreme README](https://github.com/Perkins729/OmniXtreme/blob/main/README.md#2-set-motion-path) and save it as `omnix_repo/policy/motion.npz`:

```
omnix_repo/policy/
          ├── base_policy_trt.onnx
          ├── fk_trt.onnx
          ├── motion.npz
          └── residual_policy.onnx
```

The motion path is set by `motion.motion_file_npz` in `omnix_repo/configs/config.yaml`.

## OpenVINO inference
[back to top ⬆️](#Table-of-contents:)

Run from `omnix_repo/` (the script loads `configs/config.yaml`, `policy/` and `robots/` with relative paths):

```bash
python -u deploy_mujoco.py --ov
```

- `python -u`: run Python with unbuffered stdout so progress streams live.
- `--ov`: run the FK, base and residual graphs on OpenVINO instead of ONNX Runtime.
- `--{fk,base,residual}_device`: OpenVINO device for each graph (`CPU`, `GPU`, ...).
- `--{fk,base,residual}_precision`: `INFERENCE_PRECISION_HINT` for each graph (`f16` or `f32`).
- `--cpu_threads 4`: `INFERENCE_NUM_THREADS` for graphs placed on the CPU.
- `--headless`: run the simulation without the MuJoCo viewer.

For example, to run every graph on the GPU without a viewer:

```bash
python -u deploy_mujoco.py --ov --fk_device GPU --residual_device GPU --headless
```

The robot starts from the first frame of the reference motion and tracks it until the motion ends; the viewer then closes.

With `--ov`, `deploy_mujoco.py` creates `ov.Core()` and, for each graph, calls `core.compile_model(...)` on the selected device with `{"INFERENCE_PRECISION_HINT": precision}` (plus `INFERENCE_NUM_THREADS` on the CPU) and runs it through an `OvPolicy` infer request that replaces the ONNX Runtime session or wrapper. Before compiling the FK graph, `fk_without_nonzero(...)` reads it with `core.read_model(...)` and replaces its `NonZero`/`ScatterND`/`GatherND` boolean-mask subgraph with static `Select`/`Gather` ops, so the graph has no data-dependent shapes.

## ONNX Runtime baseline
[back to top ⬆️](#Table-of-contents:)

Without `--ov`, `deploy_mujoco.py` keeps the original upstream ONNX Runtime path. Run from `omnix_repo/`:

```bash
python -u deploy_mujoco.py
```

`--headless` can be added here as well.

## Device and precision defaults
[back to top ⬆️](#Table-of-contents:)

| Graph    | Default device | Default precision |
|----------|----------------|-------------------|
| fk       | CPU            | f16               |
| base     | GPU            | f16               |
| residual | CPU            | f16               |

The flow-matching base policy is the largest graph and runs on the GPU; the small FK and residual graphs run on the CPU.